# 18 - VoxelMorph-style instance-specific

Este notebook documenta el primer experimento deep learning para registration H&E -> HSI.

La idea no es entrenar un modelo universal con 6 muestras. La idea es ajustar una red 2D tipo VoxelMorph a una muestra concreta, partiendo de la H&E ya inicializada por el baseline afin.

## Metodo

- Moving/source: mascara H&E tras registro afin clasico.
- Fixed/target: mascara HSI.
- Entrada de la red: dos mapas de distancia firmados, uno de H&E y otro de HSI.
- Salida de la red: campo de deformacion denso 2D.
- Perdida: MSE entre mapas de distancia + Dice loss de mascaras + regularizacion de suavidad.

Esto se debe defender como **deep learning instance-specific**, no como modelo generalizable.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path.cwd()
METHOD_DIR = ROOT / 'Registration' / 'DeepLearning' / 'Tecnicas_registration' / '01_voxelmorph'
SCRIPT = METHOD_DIR / 'scripts' / 'run_voxelmorph_instance.py'
print(SCRIPT)


In [ ]:
# Cambia a True si quieres relanzar los entrenamientos desde el notebook.
RUN_TRAINING = False
SUBJECTS = ['SB013', 'SB017']

if RUN_TRAINING:
    for subject in SUBJECTS:
        cmd = [sys.executable, str(SCRIPT), '--subject', subject, '--iterations', '250', '--size', '256']
        result = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True)
        print(result.stdout[-3000:])
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f'Fallo entrenamiento {subject}')


In [ ]:
summaries = []
for subject in SUBJECTS:
    out_dir = METHOD_DIR / 'outputs' / f'outputs_voxelmorph_instance_{subject}'
    summary_path = out_dir / f'{subject}_voxelmorph_instance_summary.json'
    summaries.append(json.loads(summary_path.read_text(encoding='utf-8')))
[(s['subject'], s['initial_dice'], s['final_dice']) for s in summaries]


In [ ]:
fig, axes = plt.subplots(len(SUBJECTS), 1, figsize=(16, 6 * len(SUBJECTS)))
if len(SUBJECTS) == 1:
    axes = [axes]
for ax, subject in zip(axes, SUBJECTS):
    img_path = METHOD_DIR / 'outputs' / f'outputs_voxelmorph_instance_{subject}' / f'{subject}_voxelmorph_instance_summary.png'
    ax.imshow(Image.open(img_path))
    ax.set_title(subject)
    ax.axis('off')
plt.tight_layout()


## Interpretacion

SB013 funciona como prueba de sanidad: el baseline afin ya estaba cerca y la red ajusta el contorno.

SB017 es mas interesante: el baseline afin tenia un solape bajo y la red consigue ajustar mucho mas la mascara. Aun asi, hay que revisar la deformacion visualmente porque una red instance-specific puede forzar el tejido para maximizar Dice.